Today's topics:
* `and`, `or`, and `not`
* `in`
* `while` loops
* `break` and `continue`
* nested loops

# Finding the fatigue limit

Last class we plotted one S-N campaign, and the specimen at 350 MPa was still intact when its test was stopped.

Designers often want exactly that number: the highest stress amplitude at which a steel specimen reaches ten million cycles without breaking.
It is called the **fatigue limit**.

Each test costs time.
At 50 cycles per second, a plausible testing speed, ten million cycles takes:

In [1]:
print(10000000 / 50 / 3600)  # hours for one specimen to reach ten million cycles

55.55555555555556


About 56 hours of machine time for one runout.
A lab cannot afford to test many specimens at stresses far from the limit.

The **staircase** (or up-and-down) method tests its specimens near the limit instead.
The first specimen is tested at a guessed stress.

If it breaks, the next specimen is tested one step lower.
If it runs out, the next is tested one step higher.

The stress levels move around the fatigue limit, and the results are averaged.

<!-- sketch placeholder: matsci_staircase_test_sketch (stress levels against specimen number, a step down after each break and a step up after each runout) -->

That procedure is a loop of unknown length that ends on a condition, not a loop over a list we already have.

Each decision in it combines several conditions at once.
Today's tools cover both.

We continue with campaign 29 from L15, loaded the same way:

In [2]:
import numpy as np
import pandas as pd

tests = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/fatigue_42crmo4.csv')

campaign = tests.loc[tests['curve_id'] == 29]
campaign = campaign.sort_values('stress_amplitude_MPa', ascending=False)

stress = campaign['stress_amplitude_MPa'].tolist()  # MPa
cycles = campaign['cycles'].tolist()
runout = campaign['is_runout'].tolist()             # True or False for each specimen

print(len(stress))  # specimens
print(stress)

17
[675.0, 650.0, 625.0, 575.0, 525.0, 500.0, 475.0, 462.5, 450.0, 450.0, 425.0, 425.0, 400.0, 400.0, 375.0, 375.0, 350.0]


Seventeen specimens, ordered from 675 MPa down to 350 MPa.
The three lists are parallel, so position `i` is the same specimen in all of them.

# and, or, and not

One comparison gives `True` or `False` (L11).
Specimen 8 was tested at 450 MPa and lasted 1,873,724 cycles:

In [3]:
print(stress[8] < 500)
print(cycles[8] > 1000000)

True
True


**`and`** gives `True` only when both sides are `True`.
**`or`** gives `True` when at least one side is `True`.

**`not`** flips `True` to `False` and `False` to `True`.

In [4]:
print(stress[8] < 500 and cycles[8] > 1000000)
print(stress[8] > 600 or cycles[8] > 1000000)
print(not runout[8])

True
True
True


`not runout[8]` reads as "this specimen was not a runout", which means it broke.

In a loop, `and` narrows a selection.
Here are the high-stress, short-life specimens, above 500 MPa and under 100,000 cycles:

In [5]:
count = 0

for i in range(len(stress)):
    if stress[i] > 500 and cycles[i] < 100000:
        count = count + 1
        print(stress[i], 'MPa', cycles[i], 'cycles')

print(count)

675.0 MPa 6775 cycles
650.0 MPa 15166 cycles
625.0 MPa 34667 cycles
575.0 MPa 53318 cycles
4


Four specimens meet both conditions.
The 525 MPa specimen lasted 102,666 cycles, so it fails the second one.

`or` widens a selection.
The ends of this campaign are the specimens above 600 MPa or below 400 MPa:

In [6]:
for i in range(len(stress)):
    if stress[i] > 600 or stress[i] < 400:
        print(stress[i], 'MPa')

675.0 MPa
650.0 MPa
625.0 MPa
375.0 MPa
375.0 MPa
350.0 MPa


Three high-stress specimens and three low-stress specimens.

When `not`, `and`, and `or` share a line, `not` is applied first, then `and`, then `or`.
Parentheses make the order visible, and they change it when you need a different one:

In [7]:
print(True or False and False)    # and first: True or (False and False)
print((True or False) and False)  # parentheses first

True
False


`and` and `or` each combine two single `True` or `False` values, which is the form an `if` inside a loop uses.
On a whole array they fail:

In [8]:
# EXPECTED-ERROR: this cell fails on purpose -- see the surrounding text
np.array(cycles) > 100000 and np.array(stress) > 500

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

`ValueError`: an array holds seventeen answers, and a condition has to be a single `True` or `False`.

The array version of "both" is `&`, with each comparison in parentheses, as in L14.
Inside a loop we have one specimen at a time, so `and` is the right word.

# in

**`in`** tests whether a value is present in a list.
It gives `True` or `False`, so it can go straight into an `if`.

The `surface_treatment` column records how each specimen's surface was finished.
Does the whole file include a shot-peened specimen?

In [9]:
surfaces = tests['surface_treatment'].tolist()  # all 663 specimens

print('turned' in surfaces)
print('shot peened' in surfaces)
print('shot peened' not in surfaces)

True
False
True


Turned surfaces are in the file and shot-peened surfaces are not.
`not in` is `in` followed by `not`, written as one operator.

`in` can also be tested against a short list we write ourselves.
Suppose we want only three test campaigns:

In [10]:
curve_ids = tests['curve_id'].tolist()
runouts_all = tests['is_runout'].tolist()

wanted = [29, 110, 7]

picked = 0
for i in range(len(curve_ids)):
    if curve_ids[i] in wanted:
        picked = picked + 1

print(picked)  # specimens in the three campaigns

46


That one `in` replaces `curve_ids[i] == 29 or curve_ids[i] == 110 or curve_ids[i] == 7`.
The three campaigns hold 46 specimens.

Use `.tolist()` before `in`.
On a `Series`, `in` is tested against the row labels, not the values:

In [11]:
print(100 in tests['curve_id'])             # row label 100 exists
print(100 in tests['curve_id'].tolist())    # no campaign has curve_id 100

True
False


# while

A `for` loop runs once per entry of a collection we already have.
A **`while`** loop runs as long as a condition is `True`, and the number of passes is not fixed ahead of time.

A staircase test needs this: keep stepping until the stopping condition is met.
Here is a test plan that starts at 675 MPa and steps down 25 MPa at a time while the level is at least 450 MPa:

In [12]:
level = 675

while level >= 450:
    print(level)
    level = level - 25

print('stopped at', level)

675
650
625
600
575
550
525
500
475
450
stopped at 425


The condition is checked before every pass.
When `level` reached 425 the condition became `False`, and the loop ended without running the block again.

<!-- sketch placeholder: python_while_loop_flow_sketch (check the condition; if True run the block, change the variable, check again; if False leave) -->

## The infinite loop

The line `level = level - 25` is what makes the loop end.
Without it, `level` never changes, the condition stays `True`, and the loop never ends.

The cell below is that mistake, with a second condition, `count < 5`, added as a safety limit so it stops:

In [13]:
level = 675
count = 0

while level >= 450 and count < 5:  # count < 5 is only here to rescue this demo
    print(level)
    count = count + 1              # level is never updated

675
675
675
675
675


Five identical lines, and the loop only stopped because of the safety limit.
Remove the `count < 5` condition and this loop prints forever.

If a cell of yours does not finish and its output keeps growing:
* Click the square stop button where the run button was, or choose Runtime, then Interrupt execution (shortcut Ctrl+M I).
* If the notebook still does not respond, choose Runtime, then Restart session.
  You lose the variables and have to rerun the cells above.

When you write a `while`, first find the line that makes the condition eventually `False`.

## Searching with while

Start at the highest stress, move downward one specimen at a time, and stop at the first runout.
In campaign 29, how far down do we go before the first runout?

`i` is the position we are looking at.
We keep moving to the next position while we are still inside the list and the specimen at `i` broke:

In [14]:
i = 0

while i < len(stress) and not runout[i]:
    i = i + 1

print(i, stress[i], 'MPa')

16 350.0 MPa


The first runout is at position 16, at 350 MPa.
Sixteen specimens broke before we reached it.

The order of the two conditions matters.

With `and`, the left side is evaluated first, and when it is `False` the right side is skipped.

If no specimen were a runout, `i` would reach 17 and `runout[17]` would be an `IndexError`.
The test `i < len(stress)` stops the loop first.

We can check that with the first sixteen entries, none of which is a runout:

In [15]:
broke_only = runout[:16]
j = 0

while j < len(broke_only) and not broke_only[j]:
    j = j + 1

print(j)  # j equals the length of the list: no runout found

16


`for` is the choice when we have a collection to go through.
`while` is the choice when we know the condition for stopping and not the number of passes.

### [Check your understanding]

1. Loop over positions with `range(len(stress))` and print the stress and cycles of every specimen that broke between 400 and 500 MPa, inclusive.
   Use `and`.
   Nine specimens should print.
2. Count the specimens that either ran out or lasted more than 1,000,000 cycles, using `or`.
   Then write a second version that gives the same count with `not` and `and`, and check that the two counts agree.
3. Write a `while` loop that starts at 675 MPa and subtracts 50 MPa each pass, printing the stress, and stops once the stress is 400 MPa or lower.
   Which stresses print?
4. Use `in` to print `True` or `False` for whether 450 is a stress amplitude in `stress`.

*In a staircase test, the next specimen is tested at a higher stress after which outcome, a break or a runout?
Write that rule as an `if` and an `else`.*

# break and continue

Two statements change what a loop does partway through a pass.

**`continue`** skips the rest of this pass and moves to the next entry.
**`break`** ends the loop.

## continue

What is the longest life in campaign 29?
First, tracking the largest value without any skipping:

In [16]:
longest = 0

for i in range(len(cycles)):
    if cycles[i] > longest:
        longest = cycles[i]

print(longest)

33608010


That is the runout, 33,608,010 cycles.
It is not a life (L15), so it should not count.
`continue` skips it:

In [17]:
longest = 0

for i in range(len(cycles)):
    if runout[i]:
        continue  # skip to the next specimen
    if cycles[i] > longest:
        longest = cycles[i]

print(longest)

4667019


The longest observed failure is 4,667,019 cycles, at 375 MPa.
On a runout, execution continues at the next position without reaching the comparison.

## break

Scanning from the highest stress downward, at which stress did the first specimen last more than one million cycles?

In [18]:
for i in range(len(stress)):
    if cycles[i] > 1000000:
        print(stress[i], 'MPa lasted', cycles[i], 'cycles')
        break  # no need to look at the rest

print('stopped at position', i)

450.0 MPa lasted 1873724 cycles
stopped at position 8


The answer is 450 MPa, at position 8.
The other 450 MPa specimen failed at 201,089 cycles, so scatter between specimens at the same stress is large.

Without `break`, the loop would keep going and print every later specimen above a million cycles as well.

# Nested loops and ifs

A loop body can hold another loop, and an `if` can hold another `if`.
The inner loop runs completely for every pass of the outer loop.

For each of the three campaigns in `wanted`, count its specimens and its runouts.
The outer loop runs once per campaign, and the inner loop goes through the whole table of 663 rows:

In [19]:
for c in wanted:
    n = 0
    n_runout = 0
    for i in range(len(curve_ids)):
        if curve_ids[i] == c:
            n = n + 1
            if runouts_all[i]:
                n_runout = n_runout + 1
    print('campaign', c, ':', n, 'specimens,', n_runout, 'runouts')

campaign 29 : 17 specimens, 1 runouts
campaign 110 : 16 specimens, 3 runouts
campaign 7 : 13 specimens, 3 runouts


Indentation shows the nesting.
Three outer passes with 663 inner passes each make 1989 comparisons.

The inner `if runouts_all[i]:` sits inside `if curve_ids[i] == c:`.
Two nested `if` statements with no `else` are the same as one `if` joined with `and`:

In [20]:
n_runout = 0
for i in range(len(curve_ids)):
    if curve_ids[i] == 29 and runouts_all[i]:
        n_runout = n_runout + 1

print(n_runout)  # runouts in campaign 29

1


One runout in campaign 29, matching the first line above.

A `break` inside the inner loop ends only the inner loop.
The next pass of the outer loop then handles the next entry.

## enumerate

We have been writing `for i in range(len(stress))` and then `stress[i]`.

Python has a shorthand called `enumerate` that gives the position and the value together.
It is a convenience, and `range(len(...))` always works too:

In [21]:
for i, s in enumerate(stress[:3]):
    print(i, s)

0 675.0
1 650.0
2 625.0


Each pass sets `i` to the position and `s` to the entry.
You will see this form in other people's code.

### [Check your understanding]

Each cell below runs without an error and still produces a wrong answer.

Write your prediction for each one first, then run it, then fix it.

**Bug 1.**
The goal is to count the specimens that broke at more than 500 MPa, and five specimens meet that.
What will this print?

In [ ]:
# Check 1a
count = 0

for i in range(len(stress)):
    if not runout[i] or stress[i] > 500:
        count = count + 1

print(count)

**Bug 2.**
The goal is to count the specimens that lasted more than 1,000,000 cycles, and six specimens did.
What will this print?

In [ ]:
# Check 1b
count = 0

for c in cycles:
    if c > 1000000:
        count = count + 1
        break

print(count)

**Bug 3.**
The goal is to print every stress in the test plan from 500 MPa down to 400 MPa in 25 MPa steps.
This code is shown as text because it would not finish, so do not run it.

```python
level = 500
while level >= 400:
    print(level)
level = level - 25
```

1. Why does it never end, and which stress does it print again and again?
2. Fix it by changing one line.
3. Write the fixed loop in the cell below and run it.
   Five stresses should print.

*Which of these three bugs would you notice from the output alone, and which would you only notice by checking a number you already knew?*

# Summary

On `and`, `or`, and `not`:
* `a and b` is `True` only when both are `True`, and `a or b` is `True` when at least one is.
* `not` is applied first, then `and`, then `or`, and parentheses change the order.
* `and` and `or` work on one `True` or `False` at a time, as inside a loop. Whole arrays use `&` with parentheses.

On `in`:
* `value in a_list` is `True` when the value is present, and `not in` is the opposite.
* Call `.tolist()` first, because `in` on a `Series` is tested against the row labels.

On `while`:
* `while condition:` repeats its block until the condition is `False`.
* Something inside the block has to change the condition, or the loop never ends.
* To stop a loop that never ends in Colab, click the stop button, or choose Runtime then Interrupt execution.
* `for` fits a collection we already have, and `while` fits a stopping condition.

On `break` and `continue`:
* `continue` skips to the next pass, and `break` ends the loop.
* A runout is skipped with `continue` when we measure fatigue lives.

On nesting:
* An inner loop runs completely for every pass of the outer loop.
* `enumerate(things)` gives the position and the entry together.

A fatigue limit is found by stepping the stress up and down around a stress level at which a runout occurs, which is why a staircase test is a loop with a stopping condition.
In campaign 29 the first runout is the 17th specimen, at 350 MPa.

## Further reading

* VanderPlas, *A Whirlwind Tour of Python*, "Control Flow"
* Dixon and Mood, "A Method for Obtaining and Analyzing Sensitivity Data", *Journal of the American Statistical Association* 43 (1948), the origin of the up-and-down method
* The data: FABEST benchmark fatigue dataset, [Zenodo 10.5281/zenodo.20967342](https://doi.org/10.5281/zenodo.20967342), CC BY 4.0